In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

2026-06-15 04:53:25.354250: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1781499205.554168      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1781499205.610112      58 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1781499206.081064      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781499206.081106      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781499206.081109      58 computation_placer.cc:177] computation placer alr

Num GPUs Available:  1


In [ ]:

model_path = "../input/models/saifsiddiqui123/deepfake-classifier/keras/default/1/DeepShield_EfficientNetB3.h5"

if os.path.exists(model_path):
    print("Base DeepShield model found! Loading...")
    model = tf.keras.models.load_model(model_path)
    print(f"Model Input Shape: {model.input_shape}")
else:
    print(f"Error: Could not find the model at {model_path}. Please check your dataset path in the right sidebar.")

In [ ]:
IMG_SIZE = (300, 300)
BATCH_SIZE = 32

TRAIN_DIR = "/kaggle/input/datasets/manjilkarki/deepfake-and-real-images/Dataset/Train"
VAL_DIR   = "/kaggle/input/datasets/manjilkarki/deepfake-and-real-images/Dataset/Validation"
TEST_DIR  = "/kaggle/input/datasets/manjilkarki/deepfake-and-real-images/Dataset/Test"

train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary"
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    VAL_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary"
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    shuffle=False
)

Found 140002 files belonging to 2 classes.
Found 39428 files belonging to 2 classes.
Found 10905 files belonging to 2 classes.


In [ ]:

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomBrightness(0.1)
])

train_ds = train_ds.map(
    lambda x, y: (data_augmentation(x, training=True), y),
    num_parallel_calls=tf.data.AUTOTUNE
)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)

In [ ]:
base_model = model.get_layer("efficientnetb3")
base_model.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.AUC(name="auc")]
)

print("Stage 1: Training classifier head...")
history_stage1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=2
)

In [ ]:
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

base_model = model.get_layer("efficientnetb3")
base_model.trainable = True

for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False


model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-6),
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.AUC(name="auc")]
)


callbacks = [
    EarlyStopping(
        monitor="val_auc",
        mode="max",
        patience=3,
        restore_best_weights=True
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=2,
        verbose=1
    ),
    ModelCheckpoint(
        "DeepShield_v2.keras",  
        monitor="val_auc",
        mode="max",
        save_best_only=True,
        verbose=1
    )
]

print("Stage 2: Fine-tuning base model...")
history_stage2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=4,  
    callbacks=callbacks
)

In [ ]:

best_model = tf.keras.models.load_model("../input/models/saifsiddiqui123/deepsheild-v2/keras/default/1/DeepShield_v2.keras")

loss, acc, auc = best_model.evaluate(test_ds)
print(f"\nTest Accuracy: {acc:.4f}")
print(f"Test AUC: {auc:.4f}")

I0000 00:00:1781499506.582380     126 cuda_dnn.cc:529] Loaded cuDNN version 91002


341/341 ━━━━━━━━━━━━━━━━━━━━ 61s 147ms/step - accuracy: 0.7981 - auc: 0.9231 - loss: 0.6129

Test Accuracy: 0.7981
Test AUC: 0.9231


In [ ]:
preds = best_model.predict(test_ds)


y_true = np.concatenate([y.numpy() for x, y in test_ds])
y_pred = (preds > 0.5).astype(int)

print("Classification Report:")
print(classification_report(y_true, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred))

print("ROC-AUC Score:", roc_auc_score(y_true, preds))

341/341 ━━━━━━━━━━━━━━━━━━━━ 57s 156ms/step
Classification Report:
              precision    recall  f1-score   support

         0.0       0.73      0.95      0.82      5492
         1.0       0.92      0.65      0.76      5413

    accuracy                           0.80     10905
   macro avg       0.83      0.80      0.79     10905
weighted avg       0.83      0.80      0.79     10905

Confusion Matrix:
[[5190  302]
 [1900 3513]]
ROC-AUC Score: 0.9333732023295326


In [ ]:

final_model_name = "DeepShield_EfficientNetB3_v2.keras"
best_model.save(final_model_name)
print(f"Model saved successfully as '{final_model_name}' in Kaggle Output directory.")

Model saved successfully as 'DeepShield_EfficientNetB3_v2.keras' in Kaggle Output directory.
